# Ingestion du RPG dans PostGIS

*SeineCrops — notebook d'acquisition de la donnée socle*

Ce notebook charge le **Registre parcellaire graphique (RPG)** — la vérité terrain du projet — 
dans une base **PostGIS**, de façon **rejouable** et **traçable**, dans l'esprit opérationnel inspiré du 3STR.

Il est construit pour grandir section par section. La logique lourde de chargement sera encapsulée 
dans des fonctions paramétrées idempotentes ; le notebook garde l'interactivité pour la reconnaissance, 
la visualisation et le contrôle qualité.

| # | Section | État |
|---|---|---|
| 1 | Récupération du millésime | **active** |
| 2 | Reconnaissance (couches, attributs, CRS, volume) | à venir |
| 3 | Préparation de PostGIS (base, extension, schéma `raw`) | à venir |
| 4 | Découpe spatiale sur l'AOI | à venir |
| 5 | Chargement | à venir |
| 6 | Indexation | à venir |
| 7 | Validation / QA | à venir |
| 8 | Documentation / métadonnées | à venir |

## 1 · Récupération du millésime

> **Objectif** — Identifier sans ambiguïté la source RPG retenue, la récupérer localement, 
> et en figer l'empreinte pour que la chaîne soit reproductible à l'identique.

**Offre 2024.** À partir du millésime **2024**, l'offre RPG passe d'une base unique à **sept bases 
thématiques** (huit avec les îlots, selon le descriptif de contenu v3.0), pour se conformer au règlement 
d'exécution européen 2023/138 sur les jeux de données de forte valeur. Pour SeineCrops, on retient 
**RPG Parcelles** (`PARCELLES_AGRICOLES_CONSTATEES`) : continuité directe du RPG historique 
*parcelle × culture* qui alimente la classification. (RPG PAC n'est qu'une agrégation en 4 grandes 
catégories — insuffisant pour notre usage.)

**Sens de la dépendance.** C'est la **campagne agricole / période d'observation** qui est choisie en 
premier (cf. cadrage). De cette période découlent (i) la **sélection** des acquisitions Sentinel-2 — 
échantillonnées *dans* la fenêtre, le satellite acquérant en continu — et (ii) le **millésime RPG** de 
référence, aligné sur cette même période (RPG N+1). Le millésime n'est donc **pas** « le dernier 
publié » : c'est celui qui correspond à la campagne étudiée. Le paramètre primaire est 
`ANNEE_REFERENCE` ; la fenêtre Sentinel-2 en est dérivée.

**Pourquoi épingler plutôt qu'auto-sélectionner.** Un projet auditable doit rejouer la chaîne sur 
*exactement* la même donnée. On fixe donc le millésime explicitement (et son empreinte SHA-256 en 1.2). 
Le script **interroge** néanmoins les millésimes disponibles — pour **valider** le choix et alerter s'il 
n'est pas encore publié — sans jamais le décider à la volée.

| Élément | Valeur |
|---|---|
| Produit | Registre parcellaire graphique (RPG) |
| Base retenue | **RPG Parcelles** — *parcelles agricoles constatées* |
| Millésime | défini par `ANNEE_REFERENCE` (campagne de référence) — ex. 2024, arrêté au 1ᵉʳ janv. 2025 ; disponibilité validée en ligne |
| Emprise | Normandie — unité régionale **R28** |
| Producteur / diffuseur | ASP (gestion) · IGN (production, diffusion) · MASA (cadre réglementaire) |
| Licence | **Licence Ouverte / Etalab 2.0** |
| CRS source | Lambert-93 (EPSG:2154) — *à confirmer via le descriptif de contenu* |
| Documentation | `DC_DL_RPG_3-0.pdf` (descriptif) · `SE_RPG.pdf` (suivi des évolutions) |

Portails : 
[page produit RPG (cartes.gouv.fr)](https://cartes.gouv.fr/rechercher-une-donnee/dataset/IGNF_RPG) · 
[Géoservices IGN](https://geoservices.ign.fr/rpg) · 
[data.gouv.fr](https://www.data.gouv.fr/datasets/rpg).

### 1.1 · Paramètres, période et traçabilité

On choisit la **campagne de référence** (`ANNEE_REFERENCE`) ; le millésime RPG et la fenêtre 
d'observation en découlent. On consigne ensuite une **fiche de traçabilité** auditable.

Les chemins sont ancrés sur la **racine du dépôt**, détectée en remontant jusqu'à un marqueur 
`.projectroot` (créer ce fichier vide à la racine ; `.git` sert de repli). Ainsi `data/` ne dépend 
pas du dossier d'où le notebook est lancé.

In [ ]:
from __future__ import annotations

import hashlib
import json
import urllib.request
from datetime import date
from pathlib import Path


# ── Racine du projet (indépendante du dossier d'exécution du notebook) ────
def trouver_racine(marqueurs: tuple[str, ...] = (".projectroot", ".git", "pyproject.toml")) -> Path:
    """Remonte depuis le cwd jusqu'au premier dossier contenant l'un des marqueurs.

    Évite que les chemins relatifs se résolvent par rapport à notebooks/.
    Marqueur recommandé : un fichier vide `.projectroot` à la racine du dépôt.
    """
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if any((parent / m).exists() for m in marqueurs):
            return parent
    raise FileNotFoundError(
        f"Racine introuvable (aucun marqueur {marqueurs} en remontant depuis {base})."
    )


ROOT     = trouver_racine()
DATA_DIR = ROOT / "data"

# ── Choix primaire : la campagne / le millésime de référence ─────────────
# C'est la PÉRIODE d'observation qui est choisie en premier (cf. cadrage) ;
# le millésime RPG et la sélection Sentinel-2 en DÉCOULENT.
ANNEE_REFERENCE = 2024                 # campagne agricole = millésime RPG (vérité terrain)
MILLESIME       = ANNEE_REFERENCE
REGION_CODE     = "R28"                # Normandie (code région INSEE 28)
BASE            = "PARCELLES_AGRICOLES_CONSTATEES"   # base « RPG Parcelles »
CRS_SOURCE      = "EPSG:2154"          # Lambert-93 (métropole) — à confirmer (descriptif)

# Fenêtre d'observation dérivée (sept N -> déc N+1, avec RPG = N+1) :
FENETRE_DEBUT = date(ANNEE_REFERENCE - 1, 9, 1)
FENETRE_FIN   = date(ANNEE_REFERENCE, 12, 31)
# -> Les acquisitions Sentinel-2 seront ÉCHANTILLONNÉES dans cette fenêtre
#    (le satellite acquiert en continu ; sélection hors périmètre de ce notebook).

# ── Arborescence locale, ancrée sur la racine du dépôt (schéma brut) ──────
DATA_RAW = DATA_DIR / "raw" / "rpg" / str(MILLESIME) / REGION_CODE
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Récupération de l'archive (cf. 1.2) : déposer le fichier téléchargé dans DATA_RAW,
# il y sera détecté automatiquement — aucun nom de fichier à saisir à la main.
ARCHIVE_PATTERNS = ("*.7z", "*.zip")   # motif(s) de l'archive régionale RPG
SOURCE_URL       = ""                   # FACULTATIF — vide = téléchargement manuel

print(f"Racine       : {ROOT}")
print(f"Référence    : {ANNEE_REFERENCE}  (RPG {MILLESIME}, {REGION_CODE})")
print(f"Fenêtre obs. : {FENETRE_DEBUT} -> {FENETRE_FIN}")
print(f"Dépôt local  : {DATA_RAW}")

#### Disponibilité du millésime (validation, pas auto-sélection)

On interroge le service **WFS** de la Géoplateforme et on extrait l'année **uniquement des noms de 
couches RPG**, en **affichant ces noms** pour inspection (balayer tout le `GetCapabilities` 
sur-détecterait des années d'autres jeux — voirie « Edition 2026 », BCAE…). Deux mises en garde : un 
flux peut exposer une couche **en avance** sur l'archive téléchargeable, et le RPG d'une année N n'est 
de toute façon diffusé qu'au **4ᵉ trimestre N+1**. La disponibilité réelle est donc tranchée au 
**téléchargement (1.2)**, pas par le flux. La fonction est tolérante aux pannes réseau.

In [ ]:
import re
import xml.etree.ElementTree as ET

WFS_CAPS = ("https://data.geopf.fr/wfs/ows"
            "?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetCapabilities")


def _local(tag: str) -> str:
    """Nom local d'une balise XML, sans préfixe de namespace."""
    return tag.rsplit("}", 1)[-1]


def couches_rpg_wfs(url: str = WFS_CAPS, timeout: int = 60) -> list[tuple[int, str]]:
    """Couches RPG servies par le WFS, en (millésime, nom de FeatureType).

    L'année n'est lue que dans les noms de FeatureType contenant « RPG » ou
    « parcelles_graphiques ». On renvoie les NOMS pour inspection : un flux peut
    exposer une couche en avance sur l'archive téléchargeable, et le nommage est
    en transition (v3.0). Diagnostic seulement, jamais décision.
    """
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            data = resp.read()
    except Exception as exc:  # noqa: BLE001
        print(f"Découverte indisponible ({exc!r}) — on garde le millésime pinné.")
        return []
    try:
        root = ET.fromstring(data)
    except ET.ParseError as exc:
        print(f"GetCapabilities illisible ({exc!r}) — on garde le millésime pinné.")
        return []
    res: list[tuple[int, str]] = []
    for ft in root.iter():
        if _local(ft.tag) != "FeatureType":
            continue
        nom = next((c.text for c in ft if _local(c.tag) == "Name" and c.text), None)
        if not nom or not re.match(r"(RPG\.|IGNF_RPG_)", nom, re.I):
            continue
        # N'extraire l'année QUE du préfixe (avant ":") :
        # les suffixes contiennent des dates parasites (ex. "20250621", "02-04-2026").
        prefixe = nom.split(":")[0]
        for a in re.findall(r"(20\d{2})", prefixe):
            if 2015 <= int(a) <= date.today().year + 1:
                res.append((int(a), nom))
    return sorted(set(res))


couches = couches_rpg_wfs()
dispo   = sorted({an for an, _ in couches})
if couches:
    print("Couches RPG vues dans le flux WFS (millésime : nom de couche) :")
    for an, nom in couches:
        print(f"  {an} : {nom}")
    print(f"\nMillésimes RPG en flux : {dispo}")
    print(f"{'OK ' if MILLESIME in dispo else 'ATTENTION'} millésime {MILLESIME} "
          f"{'servi' if MILLESIME in dispo else 'NON servi'} en flux.")
    print("Note : un flux peut devancer l'archive téléchargeable (placeholders, nommage v3.0) ;")
    print("       la disponibilité réelle est tranchée au téléchargement en 1.2.")
else:
    print(f"Validation en ligne ignorée — millésime retenu : {MILLESIME}.")

In [ ]:
# ── Fiche de traçabilité ─────────────────────────────────────────────────
source = {
    "produit": "Registre parcellaire graphique (RPG)",
    "base": BASE,                       # RPG Parcelles — parcelles agricoles constatées
    "annee_reference": ANNEE_REFERENCE,
    "millesime": MILLESIME,
    "fenetre_observation": [FENETRE_DEBUT.isoformat(), FENETRE_FIN.isoformat()],
    "emprise_administrative": "Normandie (R28)",
    "producteur_diffuseur": "IGN (prod./diff.), ASP (gestion), MASA (cadre réglementaire)",
    "portails": [
        "https://cartes.gouv.fr/rechercher-une-donnee/dataset/IGNF_RPG",
        "https://geoservices.ign.fr/rpg",
        "https://www.data.gouv.fr/datasets/rpg",
    ],
    "licence": "Licence Ouverte / Etalab 2.0",
    "crs_source": CRS_SOURCE,
    "documentation": {
        "descriptif_contenu": "https://data.geopf.fr/annexes/ressources/documentation/DC_DL_RPG_3-0.pdf",
        "suivi_evolutions":  "https://data.geopf.fr/annexes/ressources/documentation/SE_RPG.pdf",
    },
    "millesimes_disponibles_wfs": dispo,
    "date_recuperation": date.today().isoformat(),
    "archive": None,                    # renseigné en 1.2 (détection auto)
    "url_archive": SOURCE_URL or None,
    "sha256": None,                     # renseigné en 1.2
    "statut": "init",                   # init -> archive_recuperee / archive_absente
    "note_offre_2024": (
        "Depuis le millésime 2024, l'offre RPG comporte 7 bases thématiques "
        "(8 avec les îlots, selon le descriptif v3.0). On retient RPG Parcelles, "
        "continuité du RPG historique parcelle x culture."
    ),
}

print(json.dumps(source, indent=2, ensure_ascii=False))

### 1.2 · Récupération et empreinte

Les archives régionales RPG sont volumineuses ; le téléchargement direct (`SOURCE_URL`) est 
**facultatif** — le plus souvent on télécharge manuellement depuis la page produit et on **dépose 
l'archive dans `DATA_RAW`**, où elle est **détectée automatiquement** (pas de nom de fichier à saisir). 
On calcule alors son **empreinte SHA-256**, qui épingle le millésime exact et complète la fiche de 
traçabilité. Tant qu'aucune archive n'est présente, le `statut` de la fiche le signale explicitement.

In [ ]:
def sha256sum(path: Path, chunk: int = 1 << 20) -> str:
    """Empreinte SHA-256 d'un fichier (pinning du millésime exact)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


archive_path = None

# Téléchargement direct optionnel (si lien stable fourni et fichier encore absent).
if SOURCE_URL:
    cible = DATA_RAW / Path(SOURCE_URL).name
    if not cible.exists():
        print(f"Téléchargement -> {cible} ...")
        urllib.request.urlretrieve(SOURCE_URL, cible)  # noqa: S310
        print("Terminé.")

# Détection automatique de l'archive déposée dans DATA_RAW.
candidats = sorted({p for pat in ARCHIVE_PATTERNS for p in DATA_RAW.glob(pat)})

if len(candidats) == 1:
    archive_path = candidats[0]
    digest = sha256sum(archive_path)
    source["archive"] = archive_path.name
    source["sha256"]  = digest
    source["statut"]  = "archive_recuperee"
    print(f"OK  archive : {archive_path.name}  ({archive_path.stat().st_size / 1e6:.1f} Mo)")
    print(f"    SHA-256 : {digest}")
elif not candidats:
    source["statut"] = "archive_absente"
    print("Aucune archive détectée dans", DATA_RAW.resolve())
    print("  -> Page produit RPG : Normandie (R28), base RPG Parcelles, millésime cible ;")
    print("     déposer l'archive (.7z / .zip) dans ce dossier, puis ré-exécuter cette section.")
else:
    source["statut"] = "archive_ambigue"
    print("Plusieurs archives candidates — préciser ARCHIVE_PATTERNS :")
    for c in candidats:
        print("   -", c.name)

In [ ]:
# Persistance de la fiche de traçabilité (versionnée à côté du notebook).
source_file = DATA_RAW / "SOURCE.json"
source_file.write_text(json.dumps(source, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Fiche écrite : {source_file}  (statut : {source['statut']})")
if source["sha256"] is None:
    print("  Note : empreinte non calculée — fiche incomplète tant que l'archive n'est pas récupérée.")

### 1.3 · Référentiels associés

La donnée RPG porte des **codes** de culture, pas des libellés. Pour les sections suivantes 
(filtrage par culture, restitution), il faut la **table de correspondance** `CODE_CULTU → libellé` et 
les groupes de cultures, plus le **descriptif de contenu** qui fait foi sur le schéma attributaire v3.0. 
On réserve ici les emplacements ; la récupération et la vérification du schéma se feront en 
**section 2 (reconnaissance)**, archive décompressée.

In [ ]:
# Emplacements des référentiels (à compléter en section 2).
REF_DIR = DATA_RAW.parent / "_referentiels"
REF_DIR.mkdir(parents=True, exist_ok=True)

referentiels = {
    "codes_cultures": REF_DIR / f"REF_CULTURES_GROUPES_CULTURES_{MILLESIME}.csv",
    "descriptif_contenu": REF_DIR / "DC_DL_RPG_3-0.pdf",
}

for nom, chemin in referentiels.items():
    etat = "présent" if chemin.exists() else "à récupérer"
    print(f"{nom:20s} : {chemin.name}  ({etat})")

---
*Fin de la section 1.* La section 2 ouvrira l'archive en lecture seule (`fiona.listlayers`, `pyogrio`) 
pour inventorier couches, attributs, type de géométrie, CRS et volume — sans rien charger.

## 2 · Reconnaissance

> **Objectif** — Inventorier le contenu du GeoPackage en lecture seule : couches disponibles, 
> schéma attributaire, type de géométrie, CRS et volume. Aucune donnée n'est chargée en mémoire 
> complète ; on se donne une image fidèle de la livraison avant toute transformation.

Cette section est purement **exploratoire** : elle ne modifie ni la base PostGIS ni l'archive. 
Elle produit en revanche un **rapport de reconnaissance** (`RECON.json`) versionné à côté de 
`SOURCE.json`, qui documente ce qu'on a effectivement trouvé dans la livraison R28 2024.

**Couches attendues (v3.0, millésime 2024).** Le GeoPackage R28 devrait contenir :

| Couche | Rôle dans SeineCrops |
|---|---|
| `parcelles_graphiques` | **couche cible** — géométrie parcelle + culture principale |
| `ilots_anonymes` | contexte spatial (agrégat de parcelles d'un même exploitant) |
| `codes_cultures` | table de référence `CODE_CULTU → libellé` (sans géométrie) |

*Note v3.0.* D'autres bases thématiques (RPG PAC, RPG PP, RPG IAE…) sont diffusées séparément ; 
elles ne sont pas dans cette archive R28 et ne sont pas utiles à SeineCrops.

### Imports et paramètres

In [ ]:
from __future__ import annotations

import datetime
import json
from pathlib import Path

import fiona
import pandas as pd
import pyogrio


# ── Racine du projet ──────────────────────────────────────────────────────
def trouver_racine(marqueurs=(".projectroot", ".git", "pyproject.toml")):
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if any((parent / m).exists() for m in marqueurs):
            return parent
    raise FileNotFoundError("Racine introuvable.")


ROOT     = trouver_racine()
DATA_DIR = ROOT / "data"

MILLESIME   = 2024
REGION_CODE = "R28"
DATA_RAW    = DATA_DIR / "raw" / "rpg" / str(MILLESIME) / REGION_CODE

print(f"Racine   : {ROOT}")
print(f"DATA_RAW : {DATA_RAW}")

### 2.1 · Localisation et décompression

On localise le `.gpkg` dans `DATA_RAW`. S'il est absent mais qu'une archive `.7z` est présente, 
on la décompresse automatiquement via `py7zr`.

In [ ]:
gpkg_cible = list(DATA_RAW.rglob("RPG_Parcelles.gpkg"))

if not gpkg_cible:
    archives = sorted(DATA_RAW.glob("*.7z"))
    if not archives:
        raise FileNotFoundError(
            f"Ni RPG_Parcelles.gpkg ni .7z trouvé sous {DATA_RAW}\n"
            "-> Vérifier que DATA_RAW pointe sur le bon dossier."
        )
    import py7zr
    archive = archives[0]
    print(f"Décompression de {archive.name} ({archive.stat().st_size / 1e9:.2f} Go)…")
    with py7zr.SevenZipFile(archive, mode="r") as z:
        z.extractall(path=DATA_RAW)
    print("Terminé.")
    gpkg_cible = list(DATA_RAW.rglob("RPG_Parcelles.gpkg"))

if not gpkg_cible:
    raise FileNotFoundError("RPG_Parcelles.gpkg introuvable après décompression.")

GPKG = gpkg_cible[0]
print(f"OK  GeoPackage : {GPKG.name}")
print(f"    Chemin     : {GPKG.relative_to(DATA_RAW)}")
print(f"    Taille     : {GPKG.stat().st_size / 1e9:.2f} Go")

# Inventaire des autres bases disponibles (pour information)
print()
print("Autres bases présentes dans la livraison :")
for p in sorted(DATA_RAW.rglob("RPG_*.gpkg")):
    if p != GPKG:
        print(f"  {p.name}")

### 2.2 · Inventaire des couches

On liste toutes les couches du GeoPackage avec `fiona.listlayers` puis on inspecte pour chacune 
le type de géométrie, le CRS et le nombre d'objets — sans rien charger.

In [ ]:
couches_dispo = fiona.listlayers(GPKG)
print(f"{len(couches_dispo)} couche(s) dans {GPKG.name} :\n")

inventaire = []
for nom in couches_dispo:
    with fiona.open(GPKG, layer=nom) as src:
        entree = {
            "couche":    nom,
            "geom":      src.schema["geometry"] if src.schema["geometry"] != "None" else "—",
            "epsg":      src.crs.to_epsg() if src.crs else None,
            "n_objets":  len(src),
            "attributs": list(src.schema["properties"].keys()),
        }
        inventaire.append(entree)
        geom_str = entree["geom"] if entree["geom"] != "—" else "(table)"
        print(f"  {nom:<40s}  {geom_str:<20s}  EPSG:{entree['epsg']}"
              f"  {entree['n_objets']:>8,} objets")

# La couche cible est la première (et probablement unique) couche du fichier
COUCHE_CIBLE = couches_dispo[0]
print(f"\nCouche cible retenue : {COUCHE_CIBLE!r}")

### 2.3 · Schéma attributaire de `parcelles_graphiques`

C'est la couche cible de SeineCrops. On vérifie le schéma exact tel que livré 
— les noms d'attributs v3.0 peuvent différer légèrement de la v2.0.

Attributs attendus (descriptif de contenu v3.0) :

| Attribut | Définition | Type |
|---|---|---|
| `ID_PARCEL` | Identifiant unique de la parcelle | entier |
| `SURF_PARC` | Surface en hectares (2 décimales) | réel |
| `CODE_CULTU` | Code culture principale | texte |
| `CODE_GROUP` | Code groupe de culture principale | texte |
| `CULTURE_D1` | Code culture dérobée 1 (si applicable) | texte |
| `CULTURE_D2` | Code culture dérobée 2 (si applicable) | texte |

*Les cultures dérobées ne sont renseignées que si elles existent sur la parcelle.*

In [ ]:
with fiona.open(GPKG, layer=COUCHE_CIBLE) as src:
    schema   = src.schema
    crs      = src.crs
    n_objets = len(src)

print(f"Couche    : {COUCHE_CIBLE}")
print(f"Géométrie : {schema['geometry']}")
print(f"CRS       : EPSG:{crs.to_epsg()}")
print(f"Objets    : {n_objets:,}")
print()
print("Attributs :")
attrs_attendus = {"ID_PARCEL", "SURF_PARC", "CODE_CULTU", "CODE_GROUP", "CULTURE_D1", "CULTURE_D2"}
for attr, dtype in schema["properties"].items():
    flag = "  ← attendu" if attr in attrs_attendus else ""
    print(f"  {attr:<20s}  {dtype}{flag}")

attrs_manquants = attrs_attendus - set(schema["properties"])
if attrs_manquants:
    print(f"\nATTENTION  attributs attendus non trouvés : {attrs_manquants}")
    print("  -> Vérifier le descriptif de contenu v3.0 (renommage éventuel).")
else:
    print("\nOK  tous les attributs attendus sont présents.")

### 2.4 · Aperçu des données et distributions

On charge un **échantillon** (`pyogrio` avec `max_features`) pour inspecter les premières lignes, 
la distribution des cultures et la plausibilité des surfaces — sans charger la Normandie entière.

In [ ]:
N_SAMPLE = 50_000   # suffisant pour les distributions, léger en mémoire

df_sample = pyogrio.read_dataframe(GPKG, layer=COUCHE_CIBLE, max_features=N_SAMPLE)

print(f"Échantillon : {len(df_sample):,} parcelles (sur {n_objets:,})")
print(f"Colonnes    : {list(df_sample.columns)}")
print()
df_sample.head(5)

In [ ]:
top_cultures = (
    df_sample["code_cultu"]
    .value_counts()
    .head(20)
    .rename_axis("code_cultu")
    .reset_index(name="n_parcelles")
)
top_cultures["pct"] = (top_cultures["n_parcelles"] / len(df_sample) * 100).round(1)
print("Top 20 cultures (échantillon) :")
print(top_cultures.to_string(index=False))

In [ ]:
# ── Distribution des surfaces ─────────────────────────────────────────────
stats_surf = df_sample["surf_parc"].describe().round(3)
print("Surface (ha) — statistiques descriptives :")
print(stats_surf.to_string())
print()

grandes = df_sample[df_sample["surf_parc"] > 500]
if not grandes.empty:
    print(f"ATTENTION  {len(grandes)} parcelle(s) > 500 ha dans l'échantillon.")
else:
    print("OK  aucune parcelle > 500 ha dans l'échantillon.")

In [ ]:
df_sample = pyogrio.read_dataframe(GPKG, layer=COUCHE_CIBLE)
stats_surf = df_sample["surf_parc"].describe().round(3)
print(stats_surf.to_string())

### 2.5 · Emprise spatiale

On récupère l'emprise totale via `pyogrio.read_info` — sans charger les géométries. 
On vérifie que la Normandie est bien couverte et qu'on retrouvera l'AOI 
(Caux est / Neubourg, cross-Seine) à l'intérieur.

In [ ]:
info = pyogrio.read_info(GPKG, layer=COUCHE_CIBLE)
bbox = info["total_bounds"]   # (xmin, ymin, xmax, ymax) en unités du CRS source

print("Emprise (EPSG:2154, Lambert-93) :")
print(f"  xmin (W) : {bbox[0]:,.0f} m")
print(f"  ymin (S) : {bbox[1]:,.0f} m")
print(f"  xmax (E) : {bbox[2]:,.0f} m")
print(f"  ymax (N) : {bbox[3]:,.0f} m")
print()
print("Cohérence avec la Normandie (Lambert-93) :")
checks = [
    ("xmin > 300 000", bbox[0] > 300_000),
    ("xmax < 750 000", bbox[2] < 750_000),
    ("ymin > 6 700 000", bbox[1] > 6_700_000),
    ("ymax < 7 000 000", bbox[3] < 7_000_000),
]
for label, ok in checks:
    print(f"  {'OK ' if ok else 'KO '} {label}")

### 2.6 · Autres bases de la livraison et table de référence

La v3.0 livre les bases thématiques dans des **fichiers GeoPackage séparés** (RPG_Parcelles,
RPG_Ilots, RPG_PAC, RPG_PP, RPG_BIO, RPG_IAE, RPG_SNA, RPG_ZDH). On inventorie ici les
couches disponibles dans chaque fichier.

La table de référence `codes_cultures` (`CODE_CULTU → libellé`) est **absente de l'archive
régionale v3.0** — elle n'est plus livrée localement. On la récupère via le flux **WFS
Géoplateforme** (`RPG.2024:codes_cultures`), qui expose les 147 codes nationaux. C'est

In [ ]:
# Inventaire des bases de la livraison
print("Bases disponibles dans la livraison :")
tous_gpkg = sorted(DATA_RAW.rglob("RPG_*.gpkg"))
for p in tous_gpkg:
    couches = fiona.listlayers(p)
    print(f"  {p.name:<25s}  {len(couches)} couche(s) : {couches}")

# codes_cultures absent de l'archive v3.0 — récupération via WFS Géoplateforme
print()
print("Table codes_cultures absente de l'archive — récupération via WFS…")
url = (
    "https://data.geopf.fr/wfs/ows"
    "?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
    "&TYPENAMES=RPG.2024:codes_cultures"
    "&OUTPUTFORMAT=application%2Fjson"
)
with urllib.request.urlopen(url, timeout=60) as r:
    data = json.loads(r.read())

df_codes = pd.DataFrame([f["properties"] for f in data["features"]])
print(f"OK  {len(df_codes)} codes  |  colonnes : {list(df_codes.columns)}")
print()
print(df_codes.head(10).to_string(index=False))

# Export dans les référentiels
REF_DIR = DATA_RAW.parent / "_referentiels"
REF_DIR.mkdir(parents=True, exist_ok=True)
dest = REF_DIR / f"codes_cultures_{MILLESIME}.csv"
df_codes.to_csv(dest, index=False, encoding="utf-8")
print(f"\nExporté : {dest}")

### 2.7 · Rapport de reconnaissance

On consolide les résultats en un `RECON.json` versionné. Ce fichier complète `SOURCE.json` : 
ensemble, ils documentent *quelle donnée* a été récupérée et *ce qu'elle contient*.

In [ ]:
recon = {
    "gpkg": GPKG.name,
    "chemin_relatif": str(GPKG.relative_to(DATA_RAW)),
    "taille_go": round(GPKG.stat().st_size / 1e9, 3),
    "date_reconnaissance": datetime.date.today().isoformat(),
    "couches": inventaire,
    "RPG_Parcelles": {
        "n_objets":  n_objets,
        "geometrie": schema["geometry"],
        "epsg":      crs.to_epsg(),
        "attributs_livres": list(schema["properties"].keys()),
        "attributs_attendus_v3": [
            "id_parcel", "surf_parc", "code_cultu", "code_group",
            "culture_d1", "culture_d2", "cat_cult_p",
        ],
        "note_cat_cult_p": "Nouveauté v3.0 — catégorie PAC (TA, CP, PP, SB)",
        "bbox_lamb93": {
            "xmin": round(bbox[0]),
            "ymin": round(bbox[1]),
            "xmax": round(bbox[2]),
            "ymax": round(bbox[3]),
        },
    },
    "surf_parc_stats_completes": stats_surf.to_dict(),
    "note_surf_parc_stats": (
        "Calculé sur les 528 950 parcelles complètes — "
        "l'échantillon max_features=50 000 était biaisé vers les petites entités "
        "(tri interne du GeoPackage par id_parcel croissant)."
    ),
    "top20_cultures_sample": top_cultures.to_dict(orient="records"),
    "codes_cultures": {
        "source": "WFS Géoplateforme — RPG.2024:codes_cultures",
        "url": (
            "https://data.geopf.fr/wfs/ows"
            "?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
            "&TYPENAMES=RPG.2024:codes_cultures"
            "&OUTPUTFORMAT=application%2Fjson"
        ),
        "n_codes": len(df_codes),
        "note": (
            "Absent de l'archive régionale v3.0 — "
            "récupéré via WFS et exporté dans _referentiels/."
        ),
        "export": f"codes_cultures_{MILLESIME}.csv",
    },
    "autres_bases_livraison": [p.name for p in tous_gpkg if p != GPKG],
}

dest_recon = DATA_RAW / "RECON.json"
dest_recon.write_text(json.dumps(recon, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Rapport écrit : {dest_recon}")

---
*Fin de la section 2.* La section 3 créera la base PostGIS, activera l'extension spatiale et 
définira le schéma `raw` qui recevra `parcelles_graphiques` non transformée.